# Week 11–12: Dimensionality Reduction — Principal Component Analysis (PCA) & t-SNE

This notebook provides an in-depth, hands-on comparative exploration of:
1. **Principal Component Analysis (PCA):** Linear, global variance-maximizing orthogonal projection via SVD & Eigendecomposition.
2. **t-Distributed Stochastic Neighbor Embedding (t-SNE):** Non-linear, probabilistic local manifold learning resolving the Crowding Problem.

**Dataset:** Student Lifestyle & Academic Performance ($N = 1,000$, $d = 8$ continuous features, 3 academic tiers).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from download_dataset import load_or_generate_dataset, FEATURE_NAMES
from pca_analysis import PCAMathematicalEngine
from tsne_analysis import TSNEAnalysisEngine
from pca_vs_tsne_comparator import DimensionalityReductionComparator
import visualizer

print("Libraries imported successfully!")

## 1. Load and Inspect Dataset

In [ ]:
df = load_or_generate_dataset()
print(f"Dataset Shape: {df.shape}")
display(df.head())
print("\nCohort Distribution:")
print(df["academic_cohort"].value_counts())

## 2. Principal Component Analysis (PCA)

Standardization, Covariance Matrix, Eigendecomposition, Explained Variance Ratio, and Kaiser Criterion.

In [ ]:
X = df[FEATURE_NAMES].values
y = df["academic_cohort"].values

pca_engine = PCAMathematicalEngine()
pca_engine.fit(X, feature_names=FEATURE_NAMES)
Z_2d = pca_engine.transform(X, k=2)

print("Eigenvalues:", np.round(pca_engine.eigenvalues_, 4))
print("Explained Variance Ratio (%):", np.round(pca_engine.explained_variance_ratio_ * 100, 2))
print("Cumulative Variance (%):", np.round(pca_engine.cumulative_variance_ratio_ * 100, 2))
print(f"Kaiser Criterion (lambda >= 1.0): {pca_engine.get_kaiser_components()} components retained")

### Visualizing PCA Scree Plot & Biplot

In [ ]:
visualizer.plot_01_pca_scree_and_cumulative_variance(pca_engine)
visualizer.plot_02_pca_biplot_and_loadings(pca_engine, df, FEATURE_NAMES)
plt.show()

## 3. t-SNE Manifold Learning with Perplexity Exploration

Running t-SNE across perplexity values $\in [5, 15, 30, 50]$.

In [ ]:
tsne_engine = TSNEAnalysisEngine(perplexities=[5, 15, 30, 50], random_state=42)
embeddings = tsne_engine.fit_transform_sweep(X)

visualizer.plot_03_tsne_perplexity_exploration(embeddings, df)
plt.show()

## 4. PCA vs t-SNE 2D Projection Showdown

In [ ]:
tsne_optimal = tsne_engine.get_embedding(30)
visualizer.plot_04_pca_vs_tsne_2d_projection_showdown(Z_2d, tsne_optimal, df)
plt.show()

## 5. Quantitative Tournament Benchmark & Inverse Reconstruction

In [ ]:
comparator = DimensionalityReductionComparator(X, y)
leaderboard = comparator.run_full_tournament(Z_2d, 0.15, tsne_optimal, tsne_engine.execution_times_[30])
display(leaderboard)

rmse_dict = pca_engine.compute_reconstruction_rmse(X)
visualizer.plot_05_reconstruction_error_and_benchmark(rmse_dict, leaderboard)
plt.show()